Name: Ming Jia Zang
<br>
Github Username: mzang20
<br>
USC ID: 7754553096

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [4]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import balanced_accuracy_score, silhouette_score
from sklearn.cluster import KMeans
from joblib import Parallel, delayed

### (a) Download the Anuran Calls (MFCCs) Data Set

In [5]:
df = pd.read_csv('Data/Frogs_MFCCs.csv')
feature_cols = [c for c in df.columns if c.startswith('MFCCs')]
label_cols = ['Family', 'Genus', 'Species']

X = df[feature_cols].values
Y = df[label_cols].values
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, train_size=0.7, random_state=42)
print(X_train.shape, X_test.shape)

(5036, 22) (2159, 22)


### (b) Train a classifier for each label

#### (i) Research

Each instance has $L = 3$ labels. Let $m_i$ be the number of labels predicted incorrectly for instance $i$ (the Hamming distance between the true and predicted label triplets), over $n$ instances.

- **Exact match ratio:** fraction of instances with all labels correct, $\frac{1}{n}\sum_i \mathbb{1}[m_i = 0]$.
- **Hamming loss:** fraction of individual labels that are wrong, $\frac{1}{n}\sum_i \frac{m_i}{L}$. Lower is better.
- **Hamming score:** the multi-label accuracy from Sorower (2010), $\frac{1}{n}\sum_i \frac{|Y_i \cap Z_i|}{|Y_i \cup Z_i|}$. With the labels as sets of (label, class) pairs this is $\frac{1}{n}\sum_i \frac{L - m_i}{L + m_i}$. Higher is better.

#### (ii) Train a SVM for each of the labels

In [6]:
def evaluate(Y_true, Y_pred):
    m = (Y_true != Y_pred).sum(axis=1)
    L = Y_true.shape[1]
    scores = {'exact_match': np.mean(m == 0),
              'hamming_loss': np.mean(m / L),
              'hamming_score': np.mean((L - m) / (L + m))}
    for j, label in enumerate(label_cols):
        scores[f'balanced_acc_{label}'] = balanced_accuracy_score(Y_true[:, j], Y_pred[:, j])
    return scores


cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

def fit_per_label(model, grid):
    best, preds = {}, []
    for j, label in enumerate(label_cols):
        gs = GridSearchCV(model, grid, cv=cv, n_jobs=-1).fit(X_train, Y_train[:, j])
        best[label] = gs.best_params_
        preds.append(gs.predict(X_test))
    return pd.DataFrame(best).T, np.column_stack(preds)


C_grid = 10.0 ** np.arange(0, 5)
sigma_raw = np.round(np.arange(0.3, 1.21, 0.1), 1)
sigma_std = np.arange(2, 11, 1.0)

rbf_raw = OneVsRestClassifier(SVC(kernel='rbf'))
best_raw, pred_raw = fit_per_label(rbf_raw, {'estimator__C': C_grid,
                                             'estimator__gamma': 1 / (2 * sigma_raw ** 2)})

rbf_std = Pipeline([('scale', StandardScaler()), ('svm', OneVsRestClassifier(SVC(kernel='rbf')))])
best_std, pred_std = fit_per_label(rbf_std, {'svm__estimator__C': C_grid,
                                             'svm__estimator__gamma': 1 / (2 * sigma_std ** 2)})

print('Raw:\n', best_raw, '\n')
print('Standardized:\n', best_std, '\n')
pd.DataFrame({'raw': evaluate(Y_test, pred_raw), 'standardized': evaluate(Y_test, pred_std)})

Raw:
          estimator__C  estimator__gamma
Family          100.0             3.125
Genus            10.0             3.125
Species          10.0             3.125 

Standardized:
          svm__estimator__C  svm__estimator__gamma
Family               100.0               0.055556
Genus                 10.0               0.055556
Species               10.0               0.055556 



,raw,standardized
exact_match,0.989347,0.986568
hamming_loss,0.007565,0.010190
hamming_score,0.991478,0.988652
balanced_acc_Family,0.978238,0.978793
balanced_acc_Genus,0.961543,0.949713
balanced_acc_Species,0.968130,0.956145


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [7]:
C_l1 = 10.0 ** np.arange(-3, 3.01, 0.5)
l1_svm = Pipeline([('scale', StandardScaler()),
                   ('svm', LinearSVC(penalty='l1', dual=False, max_iter=20000, random_state=42))])
best_l1, pred_l1 = fit_per_label(l1_svm, {'svm__C': C_l1})

print(best_l1, '\n')
pd.Series(evaluate(Y_test, pred_l1), name='L1 SVM').to_frame()

             svm__C
Family    10.000000
Genus    316.227766
Species   10.000000 



,L1 SVM
exact_match,0.912459
hamming_loss,0.056971
hamming_score,0.934182
balanced_acc_Family,0.690223
balanced_acc_Genus,0.769036
balanced_acc_Species,0.894759


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [8]:
def smote(X, y, k_neighbors=5, random_state=None):
    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(y, return_counts=True)
    X_out, y_out = [X], [y]
    for c, n_c in zip(classes, counts):
        n_new = counts.max() - n_c
        if n_new == 0:
            continue
        Xc = X[y == c]
        k = min(k_neighbors, n_c - 1)
        neighbors = NearestNeighbors(n_neighbors=k + 1).fit(Xc).kneighbors(Xc, return_distance=False)[:, 1:]
        base = rng.integers(0, n_c, n_new)
        partner = neighbors[base, rng.integers(0, k, n_new)]
        X_out.append(Xc[base] + rng.random((n_new, 1)) * (Xc[partner] - Xc[base]))
        y_out.append(np.full(n_new, c, dtype=y.dtype))
    return np.vstack(X_out), np.concatenate(y_out)


class SMOTEClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, estimator=None, k_neighbors=5, random_state=None):
        self.estimator = estimator
        self.k_neighbors = k_neighbors
        self.random_state = random_state

    def fit(self, X, y):
        X_res, y_res = smote(np.asarray(X), np.asarray(y), self.k_neighbors, self.random_state)
        self.estimator_ = clone(self.estimator).fit(X_res, y_res)
        self.classes_ = self.estimator_.classes_
        return self

    def predict(self, X):
        return self.estimator_.predict(X)


l1_smote = Pipeline([('scale', StandardScaler()),
                     ('svm', SMOTEClassifier(LinearSVC(penalty='l1', dual=False, max_iter=20000, random_state=42),
                                             random_state=42))])
best_smote, pred_smote = fit_per_label(l1_smote, {'svm__estimator__C': 10.0 ** np.arange(-2, 3)})

print(best_smote, '\n')
pd.DataFrame({'Gaussian SVM (raw)': evaluate(Y_test, pred_raw),
              'Gaussian SVM (standardized)': evaluate(Y_test, pred_std),
              'L1 SVM': evaluate(Y_test, pred_l1),
              'L1 SVM + SMOTE': evaluate(Y_test, pred_smote)})

         svm__estimator__C
Family                10.0
Genus                  1.0
Species               10.0 



,Gaussian SVM (raw),Gaussian SVM (standardized),L1 SVM,L1 SVM + SMOTE
exact_match,0.989347,0.986568,0.912459,0.854099
hamming_loss,0.007565,0.010190,0.056971,0.077351
hamming_score,0.991478,0.988652,0.934182,0.903798
balanced_acc_Family,0.978238,0.978793,0.690223,0.911866
balanced_acc_Genus,0.961543,0.949713,0.769036,0.915478
balanced_acc_Species,0.968130,0.956145,0.894759,0.927462


**Conclusions:** SMOTE raises the balanced accuracy of the L1 SVM for every label (Family 0.69 to 0.91, Genus 0.77 to 0.92, Species 0.89 to 0.93), so the minority classes are recognized much better. It also pushes more majority-class instances into minority classes so exact match drops from 0.912 to 0.854 and Hamming loss rises from 0.057 to 0.077. The linear L1 SVMs are worse than the Gaussian SVMs so the best classifier overall is the Gaussian SVM on raw attributes (exact match 0.989, Hamming loss 0.008, balanced accuracy 0.96 to 0.98) which handles the imbalance well without resampling.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [9]:
X_all = df[feature_cols].values
Y_all = df[label_cols].values

def choose_k(seed):
    best_k, best_s, best_labels = None, -1, None
    for k in range(2, 51):
        labels = KMeans(n_clusters=k, n_init=1, random_state=seed).fit_predict(X_all)
        s = silhouette_score(X_all, labels, sample_size=2000, random_state=seed)
        if s > best_s:
            best_k, best_s, best_labels = k, s, labels
    return best_k, best_labels

runs = Parallel(n_jobs=-1)(delayed(choose_k)(seed) for seed in range(50))
print('k chosen in each run:', [k for k, _ in runs])

k chosen in each run: [4, 4, 4, 6, 3, 4, 4, 3, 3, 4, 4, 3, 4, 4, 3, 2, 4, 4, 2, 4, 6, 4, 4, 4, 6, 3, 4, 3, 3, 4, 2, 4, 3, 4, 4, 3, 6, 6, 2, 2, 2, 4, 4, 5, 4, 4, 3, 3, 3, 3]


### (b) Determine which family is the majority

In [10]:
def majority(labels):
    lab = pd.DataFrame(Y_all, columns=label_cols)
    lab['cluster'] = labels
    return lab.groupby('cluster')[label_cols].agg(lambda s: s.value_counts().idxmax())

majorities = [majority(labels) for _, labels in runs]
majorities[0]

,Family,Genus,Species
cluster,,,
0,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1,Hylidae,Hypsiboas,HypsiboasCordobae
2,Hylidae,Hypsiboas,HypsiboasCinerascens
3,Leptodactylidae,Adenomera,AdenomeraAndre


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [11]:
results = []
for (k, labels), maj in zip(runs, majorities):
    Y_pred = maj.loc[labels].values
    m = (Y_all != Y_pred).sum(axis=1)
    results.append({'hamming_distance': m.mean(),
                    'hamming_score': np.mean((3 - m) / (3 + m)),
                    'hamming_loss': np.mean(m / 3)})

pd.DataFrame(results).agg(['mean', 'std']).T

,mean,std
hamming_distance,0.724876,0.126789
hamming_score,0.743274,0.045164
hamming_loss,0.241625,0.042263


## 3. ISLR 12.6.2

**(a) Complete linkage:** 1 and 2 fuse at 0.3. Then $d(\{1,2\},3) = 0.5$, $d(\{1,2\},4) = 0.8$, $d(3,4) = 0.45$, so 3 and 4 fuse at 0.45. Finally $\{1,2\}$ and $\{3,4\}$ fuse at $0.8$.

```
0.80      ┌───────┴───────┐
0.45      │           ┌───┴───┐
0.30  ┌───┴───┐       │       │
      1       2       3       4
```

**(b) Single linkage:** 1 and 2 fuse at 0.3. Then $d(\{1,2\},3) = 0.4$, $d(\{1,2\},4) = 0.7$, $d(3,4) = 0.45$, so 3 joins at 0.4. Finally $d(\{1,2,3\},4) = 0.45$, so 4 joins at 0.45.

```
0.45            ┌──────┴──────┐
0.40      ┌─────┴─────┐       │
0.30  ┌───┴───┐       │       │
      1       2       3       4
```

**(c)** Cutting the complete linkage dendrogram into two clusters gives $\{1, 2\}$ and $\{3, 4\}$.

**(d)** Cutting the single linkage dendrogram into two clusters gives $\{1, 2, 3\}$ and $\{4\}$.

**(e)** Swapping the branches at each fusion of the complete linkage dendrogram gives the same fusions at the same heights so it has the same meaning:

```
0.80      ┌───────┴───────┐
0.45  ┌───┴───┐           │
0.30  │       │       ┌───┴───┐
      4       3       2       1
```